# Leica TS16 XML → Simplified + Georeferenced Export (2-step, CLI + Widgets)

This notebook is structured as requested:

- **#### Config** (all hard-set paths and defaults)
- **one code cell** with all core functions
- **#### Widgets** (interactive runner)
- **#### CLI** (command-style runner)

## Default workflow
By default it runs **2 steps**:
1. **Simplification** (XML/CSV → dense trimmed local table)
2. **Optimization / georeferencing** (GCP fit + QA + transformed export)

You can skip step 1 with a checkbox or CLI flag if you already have the simplified CSV.


#### Config

In [1]:
# ===== HARD-SET CONFIG (edit these paths first) =====

# --- Inputs ---
LEICA_INPUT = r"L:\Projects\2022-10_Project_Erkki\Data\2023-11_Merendreeburg\1_RawData\260220_TS16\200226_MEENDRE2.xml"
GCP_CSV = r"L:\Projects\2022-10_Project_Erkki\Data\2023-11_Merendreeburg\1_RawData\260220_TS16\200226_MerendreeATR.csv"

# --- Output folder ---
# None = same folder as LEICA_INPUT
OUTPUT_DIR = None

# --- Step 1 (simplification) optional explicit outputs ---
# None = auto names next to input/output dir
SIMPLIFIED_CSV_OUT = None
SIMPLIFIED_XLSX_OUT = None

# --- Step 2 (georeferenced) optional explicit outputs ---
# None = auto names next to input/output dir
GEOREF_CSV_OUT = None
GEOREF_XLSX_OUT = None

# --- Step toggles ---
RUN_STEP1_SIMPLIFY = True           # default = True
RUN_STEP2_GEOREF = True             # default = True
STEP1_EXPORT_CSV = True
STEP1_EXPORT_XLSX = True
STEP2_EXPORT_CSV = True
STEP2_EXPORT_XLSX = True

# --- GCP settings ---
GCP_IDS_EXPECTED = [str(i) for i in range(1, 9)]      # Leica local GCP labels to use
GCP_ID_REMAP = {"8": "6"}                             # known label mistake fix: local "8" corresponds to target GCP "6"
APPLY_POINT_ID_REMAP_TO_FINAL_OUTPUT = False          # usually keep raw point_oid labels in final export

# --- QA / robust fit thresholds ---
H_RESIDUAL_INLIER_THRESH_M = 0.05    # 5 cm horizontal inlier threshold for robust GCP fit
DUPLICATE_WARN_DIST_M = 0.05         # warn if two different GCP labels are within 5 cm in local coordinates

# --- CRS / height configuration ---
# Emlid coordinates in your case are already Lambert 2008 + ellipsoidal height
EMLID_INPUT_HORIZONTAL_EPSG = 3812
EMLID_INPUT_HEIGHT_MODE = "ellipsoidal"    # "ellipsoidal" or "mTAW"

# Desired final output
OUTPUT_HORIZONTAL_EPSG = 3812              # 3812 = Lambert 2008, 31370 = Lambert 72
OUTPUT_HEIGHT_MODE = "ellipsoidal"         # "ellipsoidal" or "mTAW"

# --- Belgian grid files ---
# If None, code will auto-search in OUTPUT_DIR, input folder, notebook cwd
GRID_DIR = None

# --- Diagnostics exports ---
EXPORT_DIAGNOSTICS = True                  # saves GCP fit diagnostics + duplicate summary + text report

# --- Final export columns (fixed as requested) ---
FINAL_COLUMNS = ["point_oid", "timestamp", "Easting", "Northing", "Elevation"]


#### Core Code

In [2]:
# ===== ALL CORE CODE (single cell) =====
import argparse, json, re, itertools, shlex
from pathlib import Path
from collections import defaultdict
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except Exception:
    display = print

try:
    import pyproj
    from pyproj import Transformer, datadir
    HAVE_PYPROJ = True
except Exception:
    HAVE_PYPROJ = False
    Transformer = None
    datadir = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

def _print_header(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def normalize_point_id(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    return s

def local_tag(tag: str) -> str:
    return tag.split("}")[-1] if "}" in tag else tag

def to_float(v):
    try:
        if v in (None, "", " "):
            return None
        return float(v)
    except Exception:
        return None

def parse_xyz_text(text):
    if text is None:
        return (None, None, None)
    parts = str(text).strip().split()
    vals = [to_float(p) for p in parts[:3]]
    while len(vals) < 3:
        vals.append(None)
    return tuple(vals[:3])

def cp_number(pid):
    if pid is None:
        return None
    m = re.fullmatch(r"CP\s*0*([0-9]+)", str(pid), flags=re.IGNORECASE)
    return int(m.group(1)) if m else None

def rmse(v):
    arr = np.asarray(list(v), dtype=float)
    return float(np.sqrt(np.mean(arr**2))) if arr.size else np.nan

def fit_similarity(src, dst):
    src = np.asarray(src, dtype=float)
    dst = np.asarray(dst, dtype=float)
    n, d = src.shape
    mu_s = src.mean(axis=0); mu_d = dst.mean(axis=0)
    X = src - mu_s; Y = dst - mu_d
    H = (X.T @ Y) / n
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T
    var_s = (X**2).sum() / n
    scale = float(S.sum() / var_s)
    t = mu_d - scale * (R @ mu_s)
    return scale, R, t

def apply_similarity(src, scale, R, t):
    src = np.asarray(src, dtype=float)
    return (scale * (src @ R.T)) + t

def _resolve_output_file(user_path, fallback_dir, fallback_filename):
    """
    Accept either:
    - None / empty -> use fallback_dir/fallback_filename
    - a directory path -> append fallback_filename
    - a full file path -> use it
    """
    if user_path in (None, "", "None"):
        out = Path(fallback_dir) / fallback_filename
    else:
        p = Path(str(user_path))
        # If user gave a folder (existing dir OR no suffix), treat as directory
        if p.exists() and p.is_dir():
            out = p / fallback_filename
        elif p.suffix == "":
            out = p / fallback_filename
        else:
            out = p
    out.parent.mkdir(parents=True, exist_ok=True)
    return out

def parse_leica_xml_dense(xml_path: str) -> pd.DataFrame:
    tree = ET.parse(xml_path)
    root = tree.getroot()
    raw_by_target_name = defaultdict(list)
    for ro in root.iter():
        if local_tag(ro.tag) != "RawObservation":
            continue
        target_point = None
        for ch in ro:
            if local_tag(ch.tag) == "TargetPoint":
                target_point = ch; break
        target_name = target_point.attrib.get("name") if target_point is not None else None
        raw_by_target_name[target_name].append({
            "obs_timestamp": ro.attrib.get("timeStamp"),
            "setup_id": ro.attrib.get("setupID"),
            "horiz_angle_gon": to_float(ro.attrib.get("horizAngle")),
            "zenith_angle_gon": to_float(ro.attrib.get("zenithAngle")),
            "horiz_distance_m": to_float(ro.attrib.get("horizDistance")),
            "slope_distance_m": to_float(ro.attrib.get("slopeDistance")),
        })
    rows = []

    # --- Extract InstrumentSetup point(s) as synthetic rows (so they get transformed too) ---
    # Leica/LandXML often stores the station under <InstrumentSetup><InstrumentPoint>...</InstrumentPoint>
    setup_rows = []
    for ins in root.iter():
        if local_tag(ins.tag) != "InstrumentSetup":
            continue

        setup_id_attr = ins.attrib.get("id") or ins.attrib.get("setupID") or ins.attrib.get("name")
        setup_ts = ins.attrib.get("timeStamp")

        sx = sy = sz = None
        for ch in ins:
            if local_tag(ch.tag) in ("InstrumentPoint", "InstrumentStation", "StationPoint"):
                sx, sy, sz = parse_xyz_text(ch.text)
                if sx is not None and sy is not None:
                    break

        # only add if coordinates were found
        if sx is not None and sy is not None:
            # If multiple setups exist, keep them unique
            poid = "Setup" if not setup_id_attr else f"Setup_{setup_id_attr}"
            setup_rows.append({
                "point_oid": poid,
                "point_name": poid,
                "timestamp": setup_ts,
                "x": sx, "y": sy, "z": sz,
                "setup_id": setup_id_attr,
                "horiz_distance_m": None,
                "slope_distance_m": None,
                "horiz_angle_gon": None,
                "zenith_angle_gon": None,
            })
    for cp in root.iter():
        if local_tag(cp.tag) != "CgPoint":
            continue
        point_name = cp.attrib.get("name")
        point_oid = cp.attrib.get("oID") or cp.attrib.get("oid") or cp.attrib.get("id")
        timestamp = cp.attrib.get("timeStamp")
        x, y, z = parse_xyz_text(cp.text)
        raw = raw_by_target_name[point_name].pop(0) if (point_name in raw_by_target_name and raw_by_target_name[point_name]) else None
        row = {"point_oid": point_oid, "point_name": point_name, "timestamp": timestamp, "x": x, "y": y, "z": z,
               "setup_id": None, "horiz_distance_m": None, "slope_distance_m": None, "horiz_angle_gon": None, "zenith_angle_gon": None}
        if raw: row.update(raw)
        rows.append(row)
    # prepend setup rows so they are already near the top before final sorting
    if setup_rows:
        rows = setup_rows + rows

    df = pd.DataFrame(rows)
    if "timestamp" in df.columns:
        df = df.sort_values("timestamp", na_position="last").reset_index(drop=True)
    return df

def load_leica_input(leica_input: str) -> pd.DataFrame:
    p = Path(leica_input)
    if not p.exists():
        raise FileNotFoundError(f"Leica input not found: {p}")
    if p.suffix.lower() == ".xml":
        print(f"Parsing XML: {p}")
        return parse_leica_xml_dense(str(p))
    if p.suffix.lower() == ".csv":
        print(f"Loading CSV: {p}")
        df = pd.read_csv(p)
        req = {"point_oid","timestamp","x","y","z"}
        missing = req - set(df.columns)
        if missing: raise ValueError(f"CSV missing columns: {sorted(missing)}")
        for c in ["setup_id","horiz_distance_m","slope_distance_m","horiz_angle_gon","zenith_angle_gon"]:
            if c not in df.columns: df[c] = np.nan
        return df
    raise ValueError("LEICA_INPUT must be .xml or .csv")

def reorder_points(df: pd.DataFrame, gcp_ids_expected=None):
    gcp_ids_expected = [str(x) for x in (gcp_ids_expected or [])]
    d = df.copy()
    d["point_oid_norm"] = d["point_oid"].map(normalize_point_id)

    gcp_order = {str(pid): i for i, pid in enumerate(gcp_ids_expected, start=1)}
    d["gcp_sort_idx"] = d["point_oid_norm"].map(gcp_order)
    d["cp_sort_idx"] = d["point_oid_norm"].map(cp_number)

    # Setup rows first (e.g. Setup, Setup_1, Setup_ST01)
    po = d["point_oid_norm"].fillna("").astype(str)
    d["is_setup_row"] = po.str.match(r"(?i)^setup(?:_.*)?$")

    d["sort_group"] = np.where(
        d["is_setup_row"], -1,
        np.where(
            d["gcp_sort_idx"].notna(), 0,
            np.where(d["cp_sort_idx"].notna(), 1, 2)
        )
    )

    d = d.sort_values(
        ["sort_group", "gcp_sort_idx", "cp_sort_idx", "timestamp"],
        na_position="last"
    ).reset_index(drop=True)
    return d

def step1_simplify(cfg: dict):
    _print_header("STEP 1 - Simplification")
    df = load_leica_input(cfg["LEICA_INPUT"])
    for col in ["point_oid","timestamp","x","y","z","horiz_distance_m","slope_distance_m","horiz_angle_gon","zenith_angle_gon","setup_id"]:
        if col not in df.columns: df[col] = np.nan
    df = df[["point_oid","timestamp","x","y","z","horiz_distance_m","slope_distance_m","horiz_angle_gon","zenith_angle_gon","setup_id"]].copy()
    df = reorder_points(df, cfg.get("GCP_IDS_EXPECTED", []))
    out_dir = Path(cfg["OUTPUT_DIR"]) if cfg.get("OUTPUT_DIR") else Path(cfg["LEICA_INPUT"]).parent
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(cfg["LEICA_INPUT"]).stem
    csv_out = _resolve_output_file(
        cfg.get("SIMPLIFIED_CSV_OUT"),
        out_dir,
        f"{stem}_points_dense_trimmed.csv"
    )
    xlsx_out = _resolve_output_file(
        cfg.get("SIMPLIFIED_XLSX_OUT"),
        out_dir,
        f"{stem}_points_dense_trimmed.xlsx"
    )
    if cfg.get("STEP1_EXPORT_CSV", True):
        df.to_csv(csv_out, index=False); print("Saved simplified CSV :", csv_out)
    else: csv_out = None
    if cfg.get("STEP1_EXPORT_XLSX", True):
        df.to_excel(xlsx_out, index=False); print("Saved simplified XLSX:", xlsx_out)
    else: xlsx_out = None
    print(f"Rows: {len(df):,}")
    display(df.head(20))
    return df, csv_out, xlsx_out

def load_emlid_gcps(gcp_csv_path: str) -> pd.DataFrame:
    p = Path(gcp_csv_path)
    if not p.exists(): raise FileNotFoundError(f"GCP CSV not found: {p}")
    df = pd.read_csv(p)
    low = {c.lower().strip(): c for c in df.columns}
    def find_col(cands):
        for cand in cands:
            for k,v in low.items():
                if k == cand: return v
        for cand in cands:
            for k,v in low.items():
                if cand in k: return v
        return None
    c_name = find_col(["name","point","point_id","id"]); c_e = find_col(["easting","x"]); c_n = find_col(["northing","y"]); c_z = find_col(["elevation","height","z"])
    if not all([c_name,c_e,c_n,c_z]): raise ValueError(f"Could not detect GCP columns in {p.name}. Found: {list(df.columns)}")
    out = df[[c_name,c_e,c_n,c_z]].copy(); out.columns = ["gcp_id_raw","Easting","Northing","Elevation"]
    out["gcp_id"] = out["gcp_id_raw"].map(normalize_point_id)
    for c in ["Easting","Northing","Elevation"]: out[c] = pd.to_numeric(out[c], errors="coerce")
    out = out.dropna(subset=["gcp_id","Easting","Northing","Elevation"]).reset_index(drop=True)
    return out

def summarize_local_point_duplicates(local_df: pd.DataFrame) -> pd.DataFrame:
    d = local_df.copy(); d["point_oid_norm"] = d["point_oid"].map(normalize_point_id)
    rows = []
    for pid, sub in d.groupby("point_oid_norm", dropna=False):
        xyz = sub[["x","y","z"]].apply(pd.to_numeric, errors="coerce").dropna()
        if len(xyz):
            ctr = xyz.mean(axis=0).to_numpy(); radial = np.sqrt(((xyz.to_numpy()-ctr)**2).sum(axis=1))
            spread_max = float(np.max(radial)); sx,sy,sz = xyz.std(ddof=0).tolist(); xm,ym,zm = xyz.mean().tolist()
        else:
            spread_max=sx=sy=sz=xm=ym=zm=np.nan
        rows.append({"point_oid": pid, "n_rows": len(sub), "n_measured_rows": int(pd.to_numeric(sub.get("slope_distance_m"), errors="coerce").notna().sum()),
                     "x_mean": xm, "y_mean": ym, "z_mean": zm, "spread_max_m": spread_max, "std_x_m": sx, "std_y_m": sy, "std_z_m": sz,
                     "first_timestamp": sub["timestamp"].min() if "timestamp" in sub.columns else None,
                     "last_timestamp": sub["timestamp"].max() if "timestamp" in sub.columns else None})
    return pd.DataFrame(rows).sort_values(["n_rows","point_oid"], ascending=[False,True], na_position="last").reset_index(drop=True)

def aggregate_gcp_local_points(local_df: pd.DataFrame, gcp_ids, prefer_measured=True):
    d = local_df.copy(); d["point_oid_norm"] = d["point_oid"].map(normalize_point_id)
    rows = []
    for pid in [str(x) for x in gcp_ids]:
        sub_all = d[d["point_oid_norm"] == pid].copy()
        if sub_all.empty:
            rows.append({"local_id": pid, "exists_in_leica": False}); continue
        meas_mask = pd.to_numeric(sub_all.get("slope_distance_m"), errors="coerce").notna()
        sub_used = sub_all[meas_mask].copy() if (prefer_measured and meas_mask.any()) else sub_all.copy()
        xyz_all = sub_all[["x","y","z"]].apply(pd.to_numeric, errors="coerce").dropna()
        xyz_used = sub_used[["x","y","z"]].apply(pd.to_numeric, errors="coerce").dropna()
        ctr_all = xyz_all.mean(axis=0); ctr_used = xyz_used.mean(axis=0)
        radial = np.sqrt(((xyz_all.to_numpy()-ctr_all.to_numpy())**2).sum(axis=1)) if len(xyz_all) else np.array([])
        rows.append({
            "local_id": pid, "exists_in_leica": True, "n_rows_total": int(len(sub_all)), "n_rows_measured": int(meas_mask.sum()),
            "n_rows_used": int(len(sub_used)), "used_measured_rows": bool(meas_mask.any() and prefer_measured),
            "has_horiz_distance": bool(pd.to_numeric(sub_all.get("horiz_distance_m"), errors="coerce").notna().any()),
            "has_slope_distance": bool(pd.to_numeric(sub_all.get("slope_distance_m"), errors="coerce").notna().any()),
            "has_horiz_angle": bool(pd.to_numeric(sub_all.get("horiz_angle_gon"), errors="coerce").notna().any()),
            "has_zenith_angle": bool(pd.to_numeric(sub_all.get("zenith_angle_gon"), errors="coerce").notna().any()),
            "x_local": float(ctr_used["x"]), "y_local": float(ctr_used["y"]), "z_local": float(ctr_used["z"]),
            "spread_all_rows_m": float(np.max(radial)) if radial.size else np.nan,
            "timestamp_min": sub_all["timestamp"].min() if "timestamp" in sub_all.columns else None,
            "timestamp_max": sub_all["timestamp"].max() if "timestamp" in sub_all.columns else None
        })
    return pd.DataFrame(rows)

def find_near_duplicate_gcps(gcp_local_df, thresh_m=0.05):
    d = gcp_local_df.copy()
    d = d[d["exists_in_leica"] == True].dropna(subset=["x_local","y_local","z_local"]).reset_index(drop=True)
    rows = []
    for i in range(len(d)):
        for j in range(i+1, len(d)):
            a,b = d.iloc[i], d.iloc[j]
            d3 = float(np.linalg.norm([a["x_local"]-b["x_local"], a["y_local"]-b["y_local"], a["z_local"]-b["z_local"]]))
            d2 = float(np.linalg.norm([a["x_local"]-b["x_local"], a["y_local"]-b["y_local"]]))
            if d3 <= thresh_m:
                rows.append({"local_id_a": a["local_id"], "local_id_b": b["local_id"], "local_dist_2d_m": d2, "local_dist_3d_m": d3,
                             "warning": "Different GCP labels nearly identical locally (possible label error)"})
    return pd.DataFrame(rows).sort_values("local_dist_3d_m") if rows else pd.DataFrame(columns=["local_id_a","local_id_b","local_dist_2d_m","local_dist_3d_m","warning"])

def robust_fit_local_to_global(gcp_pairs_df: pd.DataFrame, h_inlier_thresh_m=0.05):
    d = gcp_pairs_df.dropna(subset=["x_local","y_local","z_local","Easting","Northing","Elevation"]).reset_index(drop=True)
    if len(d) < 3: raise ValueError("Need at least 3 matched GCPs for a fit.")
    best = None; idx_all = list(range(len(d)))
    for swap_xy in [False, True]:
        src_all = d[["y_local","x_local"]].to_numpy() if swap_xy else d[["x_local","y_local"]].to_numpy()
        dst_all = d[["Easting","Northing"]].to_numpy()
        for r in range(3, len(d)+1):
            for subset in itertools.combinations(idx_all, r):
                src_sub = src_all[list(subset)]; dst_sub = dst_all[list(subset)]
                if np.linalg.matrix_rank(src_sub - src_sub.mean(axis=0)) < 2: continue
                try: s,R,t = fit_similarity(src_sub, dst_sub)
                except Exception: continue
                pred = apply_similarity(src_all, s, R, t)
                hres = np.sqrt(((pred - dst_all)**2).sum(axis=1))
                inliers = hres <= h_inlier_thresh_m; nin = int(inliers.sum())
                if nin == 0: continue
                score = (nin, -float(np.sqrt(np.mean(hres[inliers]**2))), -float(np.sqrt(np.mean(hres**2))), r)
                if (best is None) or (score > best["score"]):
                    best = {"score": score, "swap_xy": swap_xy, "h_scale": s, "h_R": R, "h_t": t, "hres": hres, "inliers": inliers}
    if best is None: raise RuntimeError("Could not fit a valid robust horizontal transform.")
    inl = best["inliers"]; z_src = d.loc[inl,"z_local"].to_numpy(float); z_dst = d.loc[inl,"Elevation"].to_numpy(float)
    if len(z_src) >= 2:
        A = np.vstack([z_src, np.ones_like(z_src)]).T; a,b = np.linalg.lstsq(A, z_dst, rcond=None)[0]
    else:
        a,b = 1.0, float(np.nanmean(z_dst-z_src))
    a,b = float(a), float(b)
    src_all = d[["y_local","x_local"]].to_numpy() if best["swap_xy"] else d[["x_local","y_local"]].to_numpy()
    pred_EN = apply_similarity(src_all, best["h_scale"], best["h_R"], best["h_t"]); pred_Z = d["z_local"].to_numpy(float) * a + b
    out = d.copy()
    out["pred_Easting"] = pred_EN[:,0]; out["pred_Northing"] = pred_EN[:,1]; out["pred_Elevation"] = pred_Z
    out["horizontal_residual_m"] = np.sqrt((out["pred_Easting"]-out["Easting"])**2 + (out["pred_Northing"]-out["Northing"])**2)
    out["vertical_residual_m"] = out["pred_Elevation"] - out["Elevation"]
    out["inlier_h"] = inl; out["status"] = np.where(inl, "inlier", "outlier")
    return {"pairs_table": out, "swap_xy": best["swap_xy"], "h_scale": best["h_scale"], "h_R": best["h_R"], "h_t": best["h_t"], "v_a": a, "v_b": b, "inlier_mask": inl}

def leave_one_out_checkpoints(gcp_pairs_df, inlier_mask, swap_xy_fixed):
    d = gcp_pairs_df.copy().reset_index(drop=True); d = d.loc[inlier_mask].reset_index(drop=True)
    if len(d) < 4: return pd.DataFrame(columns=["local_id","target_id","h_error_m","z_error_m"])
    rows = []
    for i in range(len(d)):
        train = d.drop(index=i).reset_index(drop=True); test = d.iloc[[i]].copy()
        src_train = train[["y_local","x_local"]].to_numpy() if swap_xy_fixed else train[["x_local","y_local"]].to_numpy()
        dst_train = train[["Easting","Northing"]].to_numpy()
        src_test = test[["y_local","x_local"]].to_numpy() if swap_xy_fixed else test[["x_local","y_local"]].to_numpy()
        dst_test = test[["Easting","Northing"]].to_numpy()
        s,R,t = fit_similarity(src_train, dst_train); pred_h = apply_similarity(src_test, s, R, t)
        h_err = float(np.linalg.norm(pred_h[0]-dst_test[0]))
        z_src = train["z_local"].to_numpy(float); z_dst = train["Elevation"].to_numpy(float)
        A = np.vstack([z_src, np.ones_like(z_src)]).T; a,b = np.linalg.lstsq(A, z_dst, rcond=None)[0]
        z_err = float(test["z_local"].iloc[0]*a + b - test["Elevation"].iloc[0])
        rows.append({"local_id": test["local_id"].iloc[0], "target_id": test["target_id"].iloc[0], "h_error_m": h_err, "z_error_m": z_err})
    return pd.DataFrame(rows).sort_values("local_id", key=lambda s: s.astype(str))

def find_grid_files(grid_dir_hint, leica_input_path, output_dir):
    dirs = []
    for c in [grid_dir_hint, output_dir, str(Path(leica_input_path).parent), str(Path.cwd())]:
        if c in (None, "", "None"): continue
        p = Path(c)
        if p.exists(): dirs.append(p)
    uniq=[]; seen=set()
    for d in dirs:
        k=str(d.resolve())
        if k not in seen: uniq.append(d); seen.add(k)
    hbg18 = None; ntv2 = None
    for d in uniq:
        if hbg18 is None and (d/"hBG18.geo").exists(): hbg18 = d/"hBG18.geo"
        if ntv2 is None and (d/"_BD72LB72_ETRS89LB08.gsb").exists(): ntv2 = d/"_BD72LB72_ETRS89LB08.gsb"
    return hbg18, ntv2, uniq

def configure_pyproj_grid_search(dirs):
    if not HAVE_PYPROJ: return
    try:
        for d in dirs: datadir.append_data_dir(str(d))
    except Exception:
        pass

def convert_belgian_output_crs(df_in, src_epsg, dst_epsg, src_h_mode, dst_h_mode, hbg18_path=None):
    src_h = str(src_h_mode).lower(); dst_h = str(dst_h_mode).lower(); df = df_in.copy()
    if int(src_epsg) == int(dst_epsg) and src_h == dst_h: return df
    if not HAVE_PYPROJ: raise RuntimeError("pyproj required for CRS/height conversion.")
    if int(src_epsg) != int(dst_epsg):
        t = Transformer.from_crs(int(src_epsg), int(dst_epsg), always_xy=True)
        e,n = t.transform(df["Easting"].to_numpy(), df["Northing"].to_numpy())
        df["Easting"], df["Northing"] = e, n
    if src_h != dst_h:
        if hbg18_path is None or not Path(hbg18_path).exists():
            raise FileNotFoundError("Height conversion requested but hBG18.geo not found.")
        t_geo = Transformer.from_crs(int(dst_epsg), 4258, always_xy=True)
        lon, lat = t_geo.transform(df["Easting"].to_numpy(), df["Northing"].to_numpy())
        if src_h == "ellipsoidal" and dst_h == "mtaw":
            pipeline = f"+proj=pipeline +step +proj=vgridshift +grids={Path(hbg18_path).as_posix()}"
        elif src_h == "mtaw" and dst_h == "ellipsoidal":
            pipeline = f"+proj=pipeline +step +inv +proj=vgridshift +grids={Path(hbg18_path).as_posix()}"
        else:
            raise ValueError("Height mode must be 'ellipsoidal' or 'mTAW'.")
        t_v = Transformer.from_pipeline(pipeline)
        _,_,z2 = t_v.transform(lon, lat, df["Elevation"].to_numpy())
        df["Elevation"] = z2
    return df

def step2_georef(cfg: dict, local_df: pd.DataFrame = None):
    _print_header("STEP 2 - Georeferencing / Optimization / QA")
    if local_df is None:
        local_df = load_leica_input(cfg["LEICA_INPUT"])
        for c in ["point_oid","timestamp","x","y","z","horiz_distance_m","slope_distance_m","horiz_angle_gon","zenith_angle_gon","setup_id"]:
            if c not in local_df.columns: local_df[c] = np.nan
        local_df = local_df[["point_oid","timestamp","x","y","z","horiz_distance_m","slope_distance_m","horiz_angle_gon","zenith_angle_gon","setup_id"]].copy()

    local_df["point_oid_norm"] = local_df["point_oid"].map(normalize_point_id)
    gcp_df = load_emlid_gcps(cfg["GCP_CSV"]); gcp_df["gcp_id"] = gcp_df["gcp_id"].map(normalize_point_id)

    print(f"Local rows: {len(local_df):,}")
    print(f"GCP rows:   {len(gcp_df):,}")

    dup_summary = summarize_local_point_duplicates(local_df)
    repeats = dup_summary[dup_summary["n_rows"] > 1]
    if len(repeats):
        print("\nRepeated point_oids (top 30):")
        display(repeats.head(30))

    gcp_local = aggregate_gcp_local_points(local_df, cfg["GCP_IDS_EXPECTED"], prefer_measured=True)
    gcp_local["target_id"] = gcp_local["local_id"].replace(cfg.get("GCP_ID_REMAP", {}))
    gcp_pairs = gcp_local.merge(gcp_df[["gcp_id","Easting","Northing","Elevation"]], left_on="target_id", right_on="gcp_id", how="left")

    print("\nGCP availability / measured-fields check:")
    display(gcp_pairs[["local_id","target_id","exists_in_leica","n_rows_total","n_rows_measured","n_rows_used","used_measured_rows",
                       "has_horiz_distance","has_slope_distance","has_horiz_angle","has_zenith_angle","x_local","y_local","z_local","spread_all_rows_m","Easting","Northing","Elevation"]])

    near_dup = find_near_duplicate_gcps(gcp_local, thresh_m=float(cfg.get("DUPLICATE_WARN_DIST_M", 0.05)))
    if len(near_dup):
        print("\n⚠️ Potential human labeling error (different GCP labels nearly identical):")
        display(near_dup)

    cand = gcp_pairs[(gcp_pairs["exists_in_leica"] == True)].dropna(subset=["Easting","Northing","Elevation"]).reset_index(drop=True)
    fit = robust_fit_local_to_global(cand, h_inlier_thresh_m=float(cfg.get("H_RESIDUAL_INLIER_THRESH_M", 0.05)))
    fit_table = fit["pairs_table"]

    print("\nChosen axis interpretation:")
    print("  local [y, x] -> [Easting, Northing]" if fit["swap_xy"] else "  local [x, y] -> [Easting, Northing]")
    print(f"Horizontal similarity scale: {fit['h_scale']:.9f}")
    print(f"Vertical model: Elevation = {fit['v_a']:.9f} * z_local + {fit['v_b']:.6f}")

    print("\nGCP residuals:")
    display(fit_table[["local_id","target_id","status","horizontal_residual_m","vertical_residual_m","Easting","Northing","Elevation","pred_Easting","pred_Northing","pred_Elevation"]].sort_values(["status","local_id"]))

    print(f"\nAccepted GCP inliers: {int(fit_table['inlier_h'].sum())}/{len(fit_table)}")
    print(f"Horizontal RMSE (inliers): {rmse(fit_table.loc[fit_table['inlier_h'],'horizontal_residual_m']):.4f} m")
    print(f"Vertical RMSE (inliers):   {rmse(fit_table.loc[fit_table['inlier_h'],'vertical_residual_m']):.4f} m")

    loo = leave_one_out_checkpoints(cand, fit["inlier_mask"], fit["swap_xy"])
    if len(loo):
        print("\nLeave-one-out checkpoint test:")
        display(loo)
        print(f"LOO horizontal RMSE: {rmse(loo['h_error_m']):.4f} m")
        print(f"LOO vertical RMSE:   {rmse(loo['z_error_m']):.4f} m")

    src = local_df[["y","x"]].to_numpy(float) if fit["swap_xy"] else local_df[["x","y"]].to_numpy(float)
    EN = apply_similarity(src, fit["h_scale"], fit["h_R"], fit["h_t"])
    Z = local_df["z"].to_numpy(float) * fit["v_a"] + fit["v_b"]

    geo_df = local_df.copy()
    geo_df["Easting"], geo_df["Northing"], geo_df["Elevation"] = EN[:,0], EN[:,1], Z

    out_dir = Path(cfg["OUTPUT_DIR"]) if cfg.get("OUTPUT_DIR") else Path(cfg["LEICA_INPUT"]).parent
    hbg18, ntv2, grid_dirs = find_grid_files(cfg.get("GRID_DIR"), cfg["LEICA_INPUT"], str(out_dir))
    if HAVE_PYPROJ: configure_pyproj_grid_search(grid_dirs)

    print("\nGrid file search:")
    for d in grid_dirs: print(" -", d)
    print("hBG18.geo:", hbg18)
    print("_BD72LB72_ETRS89LB08.gsb:", ntv2)

    geo_df = convert_belgian_output_crs(
        geo_df,
        src_epsg=int(cfg["EMLID_INPUT_HORIZONTAL_EPSG"]),
        dst_epsg=int(cfg["OUTPUT_HORIZONTAL_EPSG"]),
        src_h_mode=cfg["EMLID_INPUT_HEIGHT_MODE"],
        dst_h_mode=cfg["OUTPUT_HEIGHT_MODE"],
        hbg18_path=hbg18,
    )

    if cfg.get("APPLY_POINT_ID_REMAP_TO_FINAL_OUTPUT", False):
        geo_df["point_oid"] = geo_df["point_oid_norm"].replace(cfg.get("GCP_ID_REMAP", {}))
    else:
        geo_df["point_oid"] = geo_df["point_oid"].astype(str)

    geo_df = reorder_points(geo_df, cfg.get("GCP_IDS_EXPECTED", []))
    final_df = geo_df[["point_oid","timestamp","Easting","Northing","Elevation"]].copy()

    out_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(cfg["LEICA_INPUT"]).stem
    suffix = f"_georef_EPSG{int(cfg['OUTPUT_HORIZONTAL_EPSG'])}_{str(cfg['OUTPUT_HEIGHT_MODE'])}"
    final_csv = _resolve_output_file(
        cfg.get("GEOREF_CSV_OUT"),
        out_dir,
        f"{stem}{suffix}.csv"
    )
    final_xlsx = _resolve_output_file(
        cfg.get("GEOREF_XLSX_OUT"),
        out_dir,
        f"{stem}{suffix}.xlsx"
    )

    if cfg.get("STEP2_EXPORT_CSV", True):
        final_df.to_csv(final_csv, index=False); print("Saved transformed CSV :", final_csv)
    else: final_csv = None
    if cfg.get("STEP2_EXPORT_XLSX", True):
        final_df.to_excel(final_xlsx, index=False); print("Saved transformed XLSX:", final_xlsx)
    else: final_xlsx = None

    diag = {}
    if cfg.get("EXPORT_DIAGNOSTICS", True):
        diag["gcp_fit_csv"] = out_dir / f"{stem}_gcp_fit_diagnostics.csv"
        diag["dup_summary_csv"] = out_dir / f"{stem}_pointoid_duplicate_summary.csv"
        diag["report_txt"] = out_dir / f"{stem}_transform_report.txt"
        fit_table.to_csv(diag["gcp_fit_csv"], index=False)
        dup_summary.to_csv(diag["dup_summary_csv"], index=False)
        lines = []
        lines.append("Leica TS16 local -> georeferenced report")
        lines.append("="*60)
        lines.append(f"Leica input: {cfg['LEICA_INPUT']}")
        lines.append(f"GCP CSV: {cfg['GCP_CSV']}")
        lines.append(f"GCP remap: {cfg.get('GCP_ID_REMAP', {})}")
        lines.append("")
        lines.append("Axis interpretation:")
        lines.append("local [y,x] -> [E,N]" if fit["swap_xy"] else "local [x,y] -> [E,N]")
        lines.append(f"Horizontal scale: {fit['h_scale']:.9f}")
        lines.append(f"Vertical: Elev = {fit['v_a']:.9f} * z + {fit['v_b']:.6f}")
        lines.append("")
        lines.append(f"Inlier horizontal RMSE: {rmse(fit_table.loc[fit_table['inlier_h'],'horizontal_residual_m']):.4f} m")
        lines.append(f"Inlier vertical RMSE:   {rmse(fit_table.loc[fit_table['inlier_h'],'vertical_residual_m']):.4f} m")
        if len(loo):
            lines.append(f"LOO horizontal RMSE:    {rmse(loo['h_error_m']):.4f} m")
            lines.append(f"LOO vertical RMSE:      {rmse(loo['z_error_m']):.4f} m")
        lines.append("")
        if len(near_dup):
            lines.append("Potential near-duplicate GCP labels:")
            for _, r in near_dup.iterrows():
                lines.append(f" - {r['local_id_a']} vs {r['local_id_b']} | local 3D={r['local_dist_3d_m']:.4f} m")
        outliers = fit_table[~fit_table["inlier_h"]]
        if len(outliers):
            lines.append("Rejected GCP correspondences:")
            for _, r in outliers.iterrows():
                lines.append(f" - local {r['local_id']} -> target {r['target_id']} | h={r['horizontal_residual_m']:.3f} m, z={r['vertical_residual_m']:.3f} m")
        Path(diag["report_txt"]).write_text("\n".join(lines), encoding="utf-8")
        print("Saved diagnostics:")
        for k,v in diag.items(): print(f" - {k}: {v}")

    print("\nFinal transformed preview:")
    display(final_df.head(30))
    return {"local_df": local_df, "gcp_pairs": gcp_pairs, "fit_table": fit_table, "loo": loo, "final_df": final_df, "outputs": {"csv": final_csv, "xlsx": final_xlsx, **diag}}

def build_default_config_from_notebook_vars():
    return {
        "LEICA_INPUT": LEICA_INPUT, "GCP_CSV": GCP_CSV, "OUTPUT_DIR": OUTPUT_DIR,
        "SIMPLIFIED_CSV_OUT": SIMPLIFIED_CSV_OUT, "SIMPLIFIED_XLSX_OUT": SIMPLIFIED_XLSX_OUT,
        "GEOREF_CSV_OUT": GEOREF_CSV_OUT, "GEOREF_XLSX_OUT": GEOREF_XLSX_OUT,
        "RUN_STEP1_SIMPLIFY": RUN_STEP1_SIMPLIFY, "RUN_STEP2_GEOREF": RUN_STEP2_GEOREF,
        "STEP1_EXPORT_CSV": STEP1_EXPORT_CSV, "STEP1_EXPORT_XLSX": STEP1_EXPORT_XLSX,
        "STEP2_EXPORT_CSV": STEP2_EXPORT_CSV, "STEP2_EXPORT_XLSX": STEP2_EXPORT_XLSX,
        "GCP_IDS_EXPECTED": list(GCP_IDS_EXPECTED), "GCP_ID_REMAP": dict(GCP_ID_REMAP),
        "APPLY_POINT_ID_REMAP_TO_FINAL_OUTPUT": APPLY_POINT_ID_REMAP_TO_FINAL_OUTPUT,
        "H_RESIDUAL_INLIER_THRESH_M": H_RESIDUAL_INLIER_THRESH_M, "DUPLICATE_WARN_DIST_M": DUPLICATE_WARN_DIST_M,
        "EMLID_INPUT_HORIZONTAL_EPSG": EMLID_INPUT_HORIZONTAL_EPSG, "EMLID_INPUT_HEIGHT_MODE": EMLID_INPUT_HEIGHT_MODE,
        "OUTPUT_HORIZONTAL_EPSG": OUTPUT_HORIZONTAL_EPSG, "OUTPUT_HEIGHT_MODE": OUTPUT_HEIGHT_MODE,
        "GRID_DIR": GRID_DIR, "EXPORT_DIAGNOSTICS": EXPORT_DIAGNOSTICS, "FINAL_COLUMNS": list(FINAL_COLUMNS),
    }

def run_pipeline(cfg: dict):
    cfg = dict(cfg)
    if not cfg.get("RUN_STEP1_SIMPLIFY", True) and not cfg.get("RUN_STEP2_GEOREF", True):
        print("Nothing to run."); return None
    local_df = None
    if cfg.get("RUN_STEP1_SIMPLIFY", True):
        local_df, _, _ = step1_simplify(cfg)
    else:
        _print_header("STEP 1 - Skipped")
        print("Step 1 skipped. Step 2 will load LEICA_INPUT directly.")
    step2_result = None
    if cfg.get("RUN_STEP2_GEOREF", True):
        step2_result = step2_georef(cfg, local_df=local_df)
    else:
        _print_header("STEP 2 - Skipped")
    return {"local_df": local_df, "step2": step2_result}

def _parse_bool(v):
    if isinstance(v, bool): return v
    s = str(v).strip().lower()
    if s in ("1","true","yes","y","on"): return True
    if s in ("0","false","no","n","off"): return False
    raise argparse.ArgumentTypeError(f"Invalid bool: {v}")

def build_cli_parser():
    p = argparse.ArgumentParser(description="Leica TS16 XML -> simplified + georeferenced export (2-step)")
    p.add_argument("--leica-input", default=LEICA_INPUT)
    p.add_argument("--gcp-csv", default=GCP_CSV)
    p.add_argument("--output-dir", default=OUTPUT_DIR)
    p.add_argument("--simplified-csv-out", default=SIMPLIFIED_CSV_OUT)
    p.add_argument("--simplified-xlsx-out", default=SIMPLIFIED_XLSX_OUT)
    p.add_argument("--georef-csv-out", default=GEOREF_CSV_OUT)
    p.add_argument("--georef-xlsx-out", default=GEOREF_XLSX_OUT)
    p.add_argument("--grid-dir", default=GRID_DIR)
    p.add_argument("--skip-step1", action="store_true")
    p.add_argument("--skip-step2", action="store_true")
    p.add_argument("--step1-export-csv", type=_parse_bool, default=STEP1_EXPORT_CSV)
    p.add_argument("--step1-export-xlsx", type=_parse_bool, default=STEP1_EXPORT_XLSX)
    p.add_argument("--step2-export-csv", type=_parse_bool, default=STEP2_EXPORT_CSV)
    p.add_argument("--step2-export-xlsx", type=_parse_bool, default=STEP2_EXPORT_XLSX)
    p.add_argument("--gcp-ids", default=",".join(GCP_IDS_EXPECTED))
    p.add_argument("--gcp-remap-json", default=json.dumps(GCP_ID_REMAP))
    p.add_argument("--apply-pointid-remap-final", type=_parse_bool, default=APPLY_POINT_ID_REMAP_TO_FINAL_OUTPUT)
    p.add_argument("--h-inlier-thresh-m", type=float, default=H_RESIDUAL_INLIER_THRESH_M)
    p.add_argument("--duplicate-warn-dist-m", type=float, default=DUPLICATE_WARN_DIST_M)
    p.add_argument("--emlid-input-epsg", type=int, default=EMLID_INPUT_HORIZONTAL_EPSG)
    p.add_argument("--emlid-input-height-mode", default=EMLID_INPUT_HEIGHT_MODE)
    p.add_argument("--output-epsg", type=int, default=OUTPUT_HORIZONTAL_EPSG)
    p.add_argument("--output-height-mode", default=OUTPUT_HEIGHT_MODE)
    p.add_argument("--export-diagnostics", type=_parse_bool, default=EXPORT_DIAGNOSTICS)
    return p

def cli_args_to_cfg(args):
    gcp_ids = [normalize_point_id(x) for x in str(args.gcp_ids).split(",") if str(x).strip()]
    remap = json.loads(args.gcp_remap_json) if args.gcp_remap_json not in (None,"","None") else {}
    remap = {normalize_point_id(k): normalize_point_id(v) for k,v in dict(remap).items()}
    cfg = build_default_config_from_notebook_vars()
    cfg.update({
        "LEICA_INPUT": args.leica_input, "GCP_CSV": args.gcp_csv, "OUTPUT_DIR": _safe_path(args.output_dir),
        "SIMPLIFIED_CSV_OUT": _safe_path(args.simplified_csv_out), "SIMPLIFIED_XLSX_OUT": _safe_path(args.simplified_xlsx_out),
        "GEOREF_CSV_OUT": _safe_path(args.georef_csv_out), "GEOREF_XLSX_OUT": _safe_path(args.georef_xlsx_out), "GRID_DIR": _safe_path(args.grid_dir),
        "RUN_STEP1_SIMPLIFY": not bool(args.skip_step1), "RUN_STEP2_GEOREF": not bool(args.skip_step2),
        "STEP1_EXPORT_CSV": bool(args.step1_export_csv), "STEP1_EXPORT_XLSX": bool(args.step1_export_xlsx),
        "STEP2_EXPORT_CSV": bool(args.step2_export_csv), "STEP2_EXPORT_XLSX": bool(args.step2_export_xlsx),
        "GCP_IDS_EXPECTED": gcp_ids, "GCP_ID_REMAP": remap, "APPLY_POINT_ID_REMAP_TO_FINAL_OUTPUT": bool(args.apply_pointid_remap_final),
        "H_RESIDUAL_INLIER_THRESH_M": float(args.h_inlier_thresh_m), "DUPLICATE_WARN_DIST_M": float(args.duplicate_warn_dist_m),
        "EMLID_INPUT_HORIZONTAL_EPSG": int(args.emlid_input_epsg), "EMLID_INPUT_HEIGHT_MODE": str(args.emlid_input_height_mode),
        "OUTPUT_HORIZONTAL_EPSG": int(args.output_epsg), "OUTPUT_HEIGHT_MODE": str(args.output_height_mode),
        "EXPORT_DIAGNOSTICS": bool(args.export_diagnostics),
    })
    return cfg

def main_cli(argv=None):
    parser = build_cli_parser()
    args = parser.parse_args(argv if argv is not None else [])
    return run_pipeline(cli_args_to_cfg(args))

print("Core code loaded. Use Widgets or CLI cell.")


Core code loaded. Use Widgets or CLI cell.


#### Widgets

In [ ]:
# Interactive widgets runner (optional)
try:
    import ipywidgets as widgets
    from IPython.display import display as ipy_display, clear_output
    import json as _json

    w_leica = widgets.Text(value=str(LEICA_INPUT), description="Leica:", layout=widgets.Layout(width="95%"))
    w_gcp = widgets.Text(value=str(GCP_CSV), description="GCP CSV:", layout=widgets.Layout(width="95%"))
    w_outdir = widgets.Text(value="" if OUTPUT_DIR is None else str(OUTPUT_DIR), description="Out dir:", layout=widgets.Layout(width="95%"))

    w_skip1 = widgets.Checkbox(value=not RUN_STEP1_SIMPLIFY, description="Skip Step 1")
    w_run2 = widgets.Checkbox(value=RUN_STEP2_GEOREF, description="Run Step 2")
    w_s1_csv = widgets.Checkbox(value=STEP1_EXPORT_CSV, description="Step1 CSV")
    w_s1_xlsx = widgets.Checkbox(value=STEP1_EXPORT_XLSX, description="Step1 XLSX")
    w_s2_csv = widgets.Checkbox(value=STEP2_EXPORT_CSV, description="Step2 CSV")
    w_s2_xlsx = widgets.Checkbox(value=STEP2_EXPORT_XLSX, description="Step2 XLSX")
    w_diag = widgets.Checkbox(value=EXPORT_DIAGNOSTICS, description="Diagnostics")

    w_s1_csv_out = widgets.Text(value="" if SIMPLIFIED_CSV_OUT is None else str(SIMPLIFIED_CSV_OUT), description="S1 CSV out:", layout=widgets.Layout(width="95%"))
    w_s1_xlsx_out = widgets.Text(value="" if SIMPLIFIED_XLSX_OUT is None else str(SIMPLIFIED_XLSX_OUT), description="S1 XLSX out:", layout=widgets.Layout(width="95%"))
    w_s2_csv_out = widgets.Text(value="" if GEOREF_CSV_OUT is None else str(GEOREF_CSV_OUT), description="S2 CSV out:", layout=widgets.Layout(width="95%"))
    w_s2_xlsx_out = widgets.Text(value="" if GEOREF_XLSX_OUT is None else str(GEOREF_XLSX_OUT), description="S2 XLSX out:", layout=widgets.Layout(width="95%"))

    w_gcps = widgets.Text(value=",".join(GCP_IDS_EXPECTED), description="GCP IDs:", layout=widgets.Layout(width="60%"))
    w_remap = widgets.Text(value=_json.dumps(GCP_ID_REMAP), description="GCP remap:", layout=widgets.Layout(width="60%"))
    w_hth = widgets.FloatText(value=float(H_RESIDUAL_INLIER_THRESH_M), description="H inlier m")
    w_dup = widgets.FloatText(value=float(DUPLICATE_WARN_DIST_M), description="Dup warn m")
    w_emlid_epsg = widgets.IntText(value=int(EMLID_INPUT_HORIZONTAL_EPSG), description="Emlid EPSG")
    w_out_epsg = widgets.IntText(value=int(OUTPUT_HORIZONTAL_EPSG), description="Out EPSG")
    w_emlid_h = widgets.Dropdown(options=["ellipsoidal","mTAW"], value=str(EMLID_INPUT_HEIGHT_MODE), description="Emlid H")
    w_out_h = widgets.Dropdown(options=["ellipsoidal","mTAW"], value=str(OUTPUT_HEIGHT_MODE), description="Out H")
    w_grid_dir = widgets.Text(value="" if GRID_DIR is None else str(GRID_DIR), description="Grid dir:", layout=widgets.Layout(width="95%"))

    btn = widgets.Button(description="Run pipeline", button_style="success")
    out = widgets.Output()

    def _run(_):
        with out:
            clear_output(wait=True)
            remap = _json.loads(w_remap.value) if w_remap.value.strip() else {}
            cfg = build_default_config_from_notebook_vars()
            cfg.update({
                "LEICA_INPUT": w_leica.value.strip(),
                "GCP_CSV": w_gcp.value.strip(),
                "OUTPUT_DIR": w_outdir.value.strip() or None,
                "SIMPLIFIED_CSV_OUT": w_s1_csv_out.value.strip() or None,
                "SIMPLIFIED_XLSX_OUT": w_s1_xlsx_out.value.strip() or None,
                "GEOREF_CSV_OUT": w_s2_csv_out.value.strip() or None,
                "GEOREF_XLSX_OUT": w_s2_xlsx_out.value.strip() or None,
                "RUN_STEP1_SIMPLIFY": not w_skip1.value,
                "RUN_STEP2_GEOREF": w_run2.value,
                "STEP1_EXPORT_CSV": w_s1_csv.value,
                "STEP1_EXPORT_XLSX": w_s1_xlsx.value,
                "STEP2_EXPORT_CSV": w_s2_csv.value,
                "STEP2_EXPORT_XLSX": w_s2_xlsx.value,
                "EXPORT_DIAGNOSTICS": w_diag.value,
                "GCP_IDS_EXPECTED": [normalize_point_id(x) for x in w_gcps.value.split(",") if x.strip()],
                "GCP_ID_REMAP": {normalize_point_id(k): normalize_point_id(v) for k,v in dict(remap).items()},
                "H_RESIDUAL_INLIER_THRESH_M": float(w_hth.value),
                "DUPLICATE_WARN_DIST_M": float(w_dup.value),
                "EMLID_INPUT_HORIZONTAL_EPSG": int(w_emlid_epsg.value),
                "OUTPUT_HORIZONTAL_EPSG": int(w_out_epsg.value),
                "EMLID_INPUT_HEIGHT_MODE": w_emlid_h.value,
                "OUTPUT_HEIGHT_MODE": w_out_h.value,
                "GRID_DIR": w_grid_dir.value.strip() or None,
            })
            _ = run_pipeline(cfg)

    btn.on_click(_run)

    ipy_display(widgets.VBox([
        widgets.HTML("<b>Paths</b>"),
        w_leica, w_gcp, w_outdir,
        widgets.HTML("<b>Step toggles / exports</b>"),
        widgets.HBox([w_skip1, w_run2, w_diag]),
        widgets.HBox([w_s1_csv, w_s1_xlsx, w_s2_csv, w_s2_xlsx]),
        w_s1_csv_out, w_s1_xlsx_out, w_s2_csv_out, w_s2_xlsx_out,
        widgets.HTML("<b>GCP / QA</b>"),
        w_gcps, w_remap, widgets.HBox([w_hth, w_dup]),
        widgets.HTML("<b>CRS / Heights</b>"),
        widgets.HBox([w_emlid_epsg, w_out_epsg, w_emlid_h, w_out_h]),
        w_grid_dir,
        btn,
        out
    ]))
except Exception as e:
    print("ipywidgets not available here. Use the CLI cell below.")
    print("Reason:", e)


#### ply export

In [5]:
# ===== NEW CELL: Georef CSV -> minimal XYZ-only PLY (MeshLab-safe) =====
from pathlib import Path
import pandas as pd
import numpy as np

OUT_DIR = Path(r"C:\Users\EB\Desktop\ORBIT2.0\Orbit2\4_RawData\2_ATR\2_TS\Output")

# If None: auto-pick most recent georef CSV in OUT_DIR
GEOREF_CSV = None  # e.g. r"C:\...\200226_MEENDRE2_georef_EPSG3812_ellipsoidal.csv"

# Output (None -> same name as CSV but .ply)
PLY_OUT = None

# Optional: if you want a polyline, set True. If you just want points, keep False.
WRITE_EDGES = False


def find_latest_georef_csv(folder: Path) -> Path:
    cands = sorted(folder.glob("*_georef_EPSG*_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not cands:
        raise FileNotFoundError(f"No georef CSV found in {folder} (pattern: *_georef_EPSG*_*.csv)")
    return cands[0]


def write_xyz_ply_ascii(ply_path: Path, xyz: np.ndarray, write_edges: bool = False):
    xyz = np.asarray(xyz, dtype=float)
    if xyz.ndim != 2 or xyz.shape[1] != 3:
        raise ValueError("xyz must be an (N,3) array.")
    n = xyz.shape[0]
    m = (n - 1) if (write_edges and n >= 2) else 0

    lines = []
    lines.append("ply")
    lines.append("format ascii 1.0")
    lines.append(f"element vertex {n}")
    lines.append("property float x")
    lines.append("property float y")
    lines.append("property float z")
    if m > 0:
        lines.append(f"element edge {m}")
        lines.append("property int vertex1")
        lines.append("property int vertex2")
    lines.append("end_header")

    # vertices
    for x, y, z in xyz:
        lines.append(f"{x:.6f} {y:.6f} {z:.6f}")

    # edges (optional)
    if m > 0:
        for i in range(n - 1):
            lines.append(f"{i} {i+1}")

    ply_path.parent.mkdir(parents=True, exist_ok=True)
    ply_path.write_text("\n".join(lines), encoding="utf-8")


# --- pick input ---
src = Path(GEOREF_CSV) if GEOREF_CSV else find_latest_georef_csv(OUT_DIR)
print("Using:", src)

df = pd.read_csv(src)

# Only use Easting/Northing/Elevation and write as x/y/z
xyz = df[["Easting", "Northing", "Elevation"]].apply(pd.to_numeric, errors="coerce").dropna().to_numpy(float)
if xyz.size == 0:
    raise ValueError("No valid numeric Easting/Northing/Elevation rows found.")

ply_path = Path(PLY_OUT) if PLY_OUT else src.with_suffix(".ply")
write_xyz_ply_ascii(ply_path, xyz, write_edges=WRITE_EDGES)

print("Saved:", ply_path)
print("Vertices:", xyz.shape[0], "| Edges:", (xyz.shape[0]-1) if WRITE_EDGES and xyz.shape[0] >= 2 else 0)

Using: C:\Users\EB\Desktop\ORBIT2.0\Orbit2\4_RawData\2_ATR\2_TS\Output\200226_MEENDRE2_georef_EPSG3812_ellipsoidal.csv
Saved: C:\Users\EB\Desktop\ORBIT2.0\Orbit2\4_RawData\2_ATR\2_TS\Output\200226_MEENDRE2_georef_EPSG3812_ellipsoidal.ply
Vertices: 5668 | Edges: 0


## 1_Leica_XML

In [6]:
from pathlib import Path

# === INPUT / OUTPUT ===
# Change this to your Leica XML file
INPUT_XML = r"L:\Projects\2022-10_Project_Erkki\Data\2023-11_Merendreeburg\1_RawData\260220_TS16/200226_MEENDRE2.xml"

# Output base name (without extension). Leave None to auto-generate next to the input file.
OUTPUT_BASE = None

# Export switches
EXPORT_XLSX = True
EXPORT_CSV = True


In [7]:
import xml.etree.ElementTree as ET
from collections import defaultdict
import pandas as pd

def _local(tag: str) -> str:
    return tag.split("}")[-1] if "}" in tag else tag

def _to_float(v):
    try:
        return float(v) if v not in (None, "") else None
    except Exception:
        return None

def _parse_xyz_text(text):
    if text is None:
        return (None, None, None)
    parts = str(text).strip().split()
    vals = []
    for p in parts[:3]:
        vals.append(_to_float(p))
    while len(vals) < 3:
        vals.append(None)
    return tuple(vals[:3])

def parse_leica_xml_dense(xml_path: str) -> pd.DataFrame:
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # Collect RawObservation info by TargetPoint name (in sequence order)
    raw_by_target_name = defaultdict(list)

    for ro in root.iter():
        if _local(ro.tag) != "RawObservation":
            continue

        tp = None
        for ch in ro:
            if _local(ch.tag) == "TargetPoint":
                tp = ch
                break

        target_name = tp.attrib.get("name") if tp is not None else None

        raw_rec = {
            "setup_id": ro.attrib.get("setupID"),
            "horiz_angle_gon": _to_float(ro.attrib.get("horizAngle")),
            "zenith_angle_gon": _to_float(ro.attrib.get("zenithAngle")),
            "horiz_distance_m": _to_float(ro.attrib.get("horizDistance")),
            "slope_distance_m": _to_float(ro.attrib.get("slopeDistance")),
        }

        # Keep order in case a target name repeats
        raw_by_target_name[target_name].append(raw_rec)

    # Build rows from CgPoint and attach matching raw obs (by CgPoint name)
    rows = []
    for cp in root.iter():
        if _local(cp.tag) != "CgPoint":
            continue

        point_name = cp.attrib.get("name")
        point_oid = cp.attrib.get("oID") or cp.attrib.get("oid") or cp.attrib.get("id")
        timestamp = cp.attrib.get("timeStamp")
        x, y, z = _parse_xyz_text(cp.text)

        raw_match = None
        if point_name in raw_by_target_name and raw_by_target_name[point_name]:
            raw_match = raw_by_target_name[point_name].pop(0)

        row = {
            "point_oid": point_oid,
            "timestamp": timestamp,
            "x": x,
            "y": y,
            "z": z,
            "horiz_distance_m": None,
            "slope_distance_m": None,
            "horiz_angle_gon": None,
            "zenith_angle_gon": None,
            "setup_id": None,
        }

        if raw_match:
            row.update(raw_match)

        rows.append(row)

    df = pd.DataFrame(rows)

    # Optional: sort by timestamp (string sort works for this ISO-like format)
    if "timestamp" in df.columns:
        df = df.sort_values("timestamp", na_position="last").reset_index(drop=True)

    return df


In [8]:
# Run conversion
xml_path = Path(INPUT_XML)
if not xml_path.exists():
    raise FileNotFoundError(f"Input file not found: {xml_path}")

if OUTPUT_BASE is None:
    out_base = xml_path.with_suffix("")
else:
    out_base = Path(OUTPUT_BASE)

df = parse_leica_xml_dense(str(xml_path))

# Save
xlsx_path = str(out_base) + "_points_dense_trimmed.xlsx"
csv_path = str(out_base) + "_points_dense_trimmed.csv"

if EXPORT_XLSX:
    df.to_excel(xlsx_path, index=False)
if EXPORT_CSV:
    df.to_csv(csv_path, index=False)

print(f"Rows: {len(df)}")
print(f"Measured rows with distances: {df['slope_distance_m'].notna().sum()}")
print("Columns:", list(df.columns))
print("XLSX:", xlsx_path if EXPORT_XLSX else "(skipped)")
print("CSV :", csv_path if EXPORT_CSV else "(skipped)")

df.head(10)


Rows: 5667
Measured rows with distances: 5659
Columns: ['point_oid', 'timestamp', 'x', 'y', 'z', 'horiz_distance_m', 'slope_distance_m', 'horiz_angle_gon', 'zenith_angle_gon', 'setup_id']
XLSX: L:\Projects\2022-10_Project_Erkki\Data\2023-11_Merendreeburg\1_RawData\260220_TS16\200226_MEENDRE2_points_dense_trimmed.xlsx
CSV : L:\Projects\2022-10_Project_Erkki\Data\2023-11_Merendreeburg\1_RawData\260220_TS16\200226_MEENDRE2_points_dense_trimmed.csv


,point_oid,timestamp,x,y,z,horiz_distance_m,slope_distance_m,horiz_angle_gon,zenith_angle_gon,setup_id
0,2.10000,2026-02-20T11:55:15.29,1034.879295,1005.727590,0.024370,NaN,NaN,NaN,NaN,None
1,1,2026-02-20T11:57:17.98,1014.436777,995.166239,0.079361,NaN,NaN,NaN,NaN,None
2,1,2026-02-20T11:58:14.36,1014.436632,995.166192,0.079607,23.009534,23.0096,230.358363,99.847953,TPSSetupID_3_2
3,1,2026-02-20T11:58:14.36,1014.436488,995.166144,0.079853,NaN,NaN,NaN,NaN,None
4,2,2026-02-20T11:59:03.27,1028.497000,1009.874290,-1.956322,NaN,NaN,NaN,NaN,None
5,3,2026-02-20T11:59:17.04,1028.495928,1009.875114,-1.956700,7.612446,7.8660,163.318482,116.207974,TPSSetupID_3_2
6,2,2026-02-20T11:59:37.55,1028.495789,1009.875258,-1.956748,NaN,NaN,NaN,NaN,None
7,2,2026-02-20T12:00:01.00,1028.496354,1009.874829,-1.956544,7.611098,7.8646,163.319378,116.207763,TPSSetupID_3_2
8,2,2026-02-20T12:00:01.00,1028.496272,1009.874941,-1.956562,NaN,NaN,NaN,NaN,None
9,5,2026-02-20T12:02:31.35,1038.239307,966.682506,6.213380,NaN,NaN,NaN,NaN,None


In [ ]:
python MRK2RC.py ^
  --roots ^
    "L:\Projects\2022-10_Project_Erkki\Data\2023-11_Merendreeburg\1_RawData\260220\DJI_202602201316_684_05MerendreebrugUnderdeckFlightRouteSection2" ^
    "L:\Projects\2022-10_Project_Erkki\Data\2023-11_Merendreeburg\1_RawData\260220\DJI_202602201224_681_OverviewAuto" ^
  --out_root "L:\Projects\2022-10_Project_Erkki\Data\2023-11_Merendreeburg\1_RawData\260220\99_RealityScanTrajectory" ^
  --rsproj "L:\Projects\2022-10_Project_Erkki\Data\2023-11_Merendreeburg\3_Processing\260220_Merendree_Auto1_redo_REALITYSCAN_copy.rsproj" ^
  --crs_out "EPSG:3812" ^
  --min_height_m 60 ^
  --max_sig_e 0.03 ^
  --max_sig_n 0.03 ^
  --max_sig_v 0.06 ^
  --enforce_single_calibration_and_lens_group